In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
from scipy.stats import norm
from scipy.optimize import minimize
from itertools import product

# ----------------------------
# Load data
# ----------------------------
X = np.load(r"C:\Users\nmnko\Desktop\NEW ML 2025\Module 12\Data\initial_data for functions Mini lesson 12.8\function_8\initial_inputs.npy")
y = np.load(r"C:\Users\nmnko\Desktop\NEW ML 2025\Module 12\Data\initial_data for functions Mini lesson 12.8\function_8\initial_outputs.npy")
print(f"Inputs:{X}")
print(f"Outputs:{y}")

Inputs:[[0.60499445 0.29221502 0.90845275 0.35550624 0.20166872 0.57533801
  0.31031095 0.73428138]
 [0.17800696 0.56622265 0.99486184 0.21032501 0.32015266 0.70790879
  0.63538449 0.10713163]
 [0.00907698 0.81162615 0.52052036 0.07568668 0.26511183 0.09165169
  0.59241515 0.36732026]
 [0.50602816 0.65373012 0.36341078 0.17798105 0.0937283  0.19742533
  0.7558269  0.29247234]
 [0.35990926 0.24907568 0.49599717 0.70921498 0.11498719 0.28920692
  0.55729515 0.59388173]
 [0.77881834 0.0034195  0.33798313 0.51952778 0.82090699 0.53724669
  0.5513471  0.66003209]
 [0.90864932 0.0622497  0.23825955 0.76660355 0.13233596 0.99024381
  0.68806782 0.74249594]
 [0.58637144 0.88073573 0.74502075 0.54603485 0.00964888 0.74899176
  0.23090707 0.09791562]
 [0.76113733 0.85467239 0.38212433 0.33735198 0.68970832 0.30985305
  0.63137968 0.04195607]
 [0.9849332  0.69950626 0.9988855  0.18014846 0.58014315 0.23108719
  0.49082694 0.31368272]
 [0.11207131 0.43773566 0.59659878 0.59277563 0.22698177 0.4101

In [9]:
y_best = np.max(y)

# ----------------------------
# Acquisition functions
# ----------------------------
def expected_improvement(gp, y_best, x):
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    s = float(sigma[0])
    if s <= 1e-12:
        return 0.0
    Z = (mu[0] - y_best) / s
    return float((mu[0] - y_best) * norm.cdf(Z) + s * norm.pdf(Z))

def probability_improvement(gp, y_best, x, xi=0.0):
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    s = float(sigma[0])
    if s <= 1e-12:
        return 0.0
    Z = (mu[0] - y_best - xi) / s
    return float(norm.cdf(Z))

def upper_confidence_bound(gp, x, beta=1.0):
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    return float(mu[0] + beta * sigma[0])


# Bounds from data
def get_bounds_from_data(X, margin=0.01):
    X = np.array(X)
    lower = np.maximum(X.min(axis=0) - margin, 0.0)  # ✅ clip at 0
    upper = X.max(axis=0) + margin
    return [(float(l), float(u)) for l, u in zip(lower, upper)]


# Acquisition optimizer
def optimize_acquisition_max(acq_func, gp, X, y_best, n_restarts=50, margin=0.01):
    rng = np.random.RandomState(42)
    X = np.array(X)
    dim = X.shape[1]
    bounds = get_bounds_from_data(X, margin)

    best_x = None
    best_val = -np.inf

    #Random starts
    starts = list(rng.rand(n_restarts, dim) * (np.array(bounds)[:, 1] - np.array(bounds)[:, 0]) 
                  + np.array(bounds)[:, 0])

    #Existing observations
    starts.extend(list(X))

    #Corners
    lower = np.array([b[0] for b in bounds])
    upper = np.array([b[1] for b in bounds])
    corners = list(product(*zip(lower, upper)))
    starts.extend(corners)

    #Optimizer
    for x0 in starts:
        x0 = np.array(x0)
        res = minimize(lambda xx: -acq_func(gp, y_best, xx),
                       x0,
                       bounds=bounds,
                       method='L-BFGS-B')
        if not res.success:
            continue
        val = -res.fun
        if val > best_val:
            best_val = val
            best_x = res.x

    return best_x, best_val

# Kernels

kernel_list = {
    "RBF": C(1.0, (1e-3, 1e3)) * RBF(length_scale=0.1, length_scale_bounds=(1e-2, 10)) 
           + WhiteKernel(noise_level=0.5, noise_level_bounds=(0.01, 1.0)),

    "Matern": C(1.0, (1e-3, 1e3)) * Matern(length_scale=0.1, length_scale_bounds=(1e-2, 10), nu=2.5) 
              + WhiteKernel(noise_level=0.5, noise_level_bounds=(0.01, 1.0)),

    "RationalQuadratic": C(1.0, (1e-3, 1e3)) * RationalQuadratic(length_scale=0.1, alpha=1.0, 
                                                                  length_scale_bounds=(1e-2, 10),
                                                                  alpha_bounds=(0.1, 10)) 
                        + WhiteKernel(noise_level=0.5, noise_level_bounds=(0.01, 1.0))
}

# Loop over kernels

for name, kernel in kernel_list.items():
    gp = GaussianProcessRegressor(kernel=kernel,
                                  n_restarts_optimizer=50,
                                  normalize_y=True,
                                  random_state=42)
    gp.fit(X, y)

    print(f"\nKernel: {name}, Optimized kernel: {gp.kernel_}")
    y_best = np.max(y)


    x_ei, v_ei   = optimize_acquisition_max(expected_improvement, gp, X, y_best=y_best)
    x_pi, v_pi   = optimize_acquisition_max(probability_improvement, gp, X, y_best=y_best)
    x_ucb, v_ucb = optimize_acquisition_max(lambda g,y,x: upper_confidence_bound(g, x, beta=2.5),
                                            gp, X, y_best=y_best)

    print("Next points and values:")
    print("EI :", x_ei, "Value:", v_ei)
    print("PI :", x_pi, "Value:", v_pi)
    print("UCB:", x_ucb, "Value:", v_ucb)
    print("y mean, std:", np.mean(y), np.std(y))
    print(gp.kernel_)


Kernel: RBF, Optimized kernel: 2.02**2 * RBF(length_scale=1.5) + WhiteKernel(noise_level=0.0172)
Next points and values:
EI : [0.03111971 0.2245557  0.01292868 0.09905603 0.996902   0.32514424
 0.02590888 0.47140712] Value: 0.9300712886537202
PI : [0.08350771 0.28507283 0.06483338 0.27510016 0.6924047  0.48125737
 0.23325327 0.54487861] Value: 0.9935085713995767
UCB: [0.04698295 0.         0.01292868 0.         0.996902   0.01211341
 0.02590888 0.03195607] Value: 12.199303741515159
y mean, std: 7.815274458164078 0.9469034630327874
2.02**2 * RBF(length_scale=1.5) + WhiteKernel(noise_level=0.0172)


C:\Users\nmnko\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Kernel: Matern, Optimized kernel: 3.96**2 * Matern(length_scale=3.53, nu=2.5) + WhiteKernel(noise_level=0.01)
Next points and values:
EI : [0.         0.2226331  0.01292868 0.18105366 0.9045576  0.46290099
 0.02590888 0.4850674 ] Value: 0.800740115013269
PI : [0.06900396 0.27554961 0.06121019 0.29032437 0.62328399 0.51619822
 0.24472279 0.55707759] Value: 0.9972540356333573
UCB: [0.         0.         0.01292868 0.         0.996902   0.01211341
 0.02590888 0.03195607] Value: 11.853799356359092
y mean, std: 7.815274458164078 0.9469034630327874
3.96**2 * Matern(length_scale=3.53, nu=2.5) + WhiteKernel(noise_level=0.01)


C:\Users\nmnko\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Kernel: RationalQuadratic, Optimized kernel: 7.95**2 * RationalQuadratic(alpha=0.133, length_scale=5.73) + WhiteKernel(noise_level=0.01)
Next points and values:
EI : [0.         0.26900066 0.01292868 0.22987204 0.73049888 0.49358386
 0.05915792 0.49333984] Value: 0.7080533653863943
PI : [0.08892594 0.2683045  0.0793579  0.28631569 0.58596961 0.52054048
 0.27253669 0.5705938 ] Value: 0.9990907202298516
UCB: [0.         0.         0.01292868 0.         0.996902   0.01211341
 0.02590888 0.03195607] Value: 11.452874760105573
y mean, std: 7.815274458164078 0.9469034630327874
7.95**2 * RationalQuadratic(alpha=0.133, length_scale=5.73) + WhiteKernel(noise_level=0.01)
